# 📗 슈퍼바이저와 워커

에이전트 하나가 모든 일을 맡으면 도구와 지시가 늘수록 판단(예: 지금 어떤 도구를 부를지)이 흐려집니다. 이 교안에서는 한 가지 일을 맡는 **워커**와, 워커의 보고를 읽고 다음 일을 정하는 **슈퍼바이저**를 만듭니다.

**학습 목표**

1. 라우터와 슈퍼바이저의 차이를 설명합니다.
2. `Command(goto, update)`로 노드가 다음 노드와 State에 쓸 값을 함께 정하게 합니다.
3. 서로 다른 도구를 가진 워커를 슈퍼바이저 그래프로 묶고, 처리 상한에 닿으면 상담원에게 넘깁니다.
4. 워커를 도구(`ask_worker`)로 부르는 supervisor 에이전트를 만들고, supervisor 노드 방식과 비교합니다.

## 1. 왜 여러 에이전트로 나눌까

### 1-1. 혼자 다 하기 vs 전문가 팀

**멀티 에이전트**는 역할이 다른 에이전트 여럿이 한 업무(예: 9월 콘텐츠 유입이 30% 줄어든 원인을 찾아 보고서 쓰기)를 나눠 처리하는 구조입니다. 에이전트마다 도구와 지시가 적어서 맡은 일에 집중합니다.

이 콘텐츠 분석 팀은 종합실습01 프로젝트(`종합실습01/starter/README.md`)에서 직접 만들고, 이 교안에서는 더 작은 쇼핑몰 CS 팀으로 구조를 먼저 익힙니다.

<img src="images/01_single_vs_team.png" width="1050" alt="9월 유입 30% 하락 원인 보고서를 에이전트 하나가 데이터베이스·코드·PDF·웹 검색·보고서 도구를 모두 붙들고 처리하는 모습과, 슈퍼바이저가 데이터 분석가·문서 검색 담당·웹 조사·보고서 작성자에게 각자 맡은 도구만 주고 나눠 맡기는 모습의 비교">

### 1-2. 나누지 않는 편이 나을 때

- 도구가 몇 개뿐이고 에이전트 하나로 잘 되는 일
- 응답 속도와 비용이 중요한 일. 에이전트가 늘면 LLM 호출도 늘어납니다.
- 단계가 늘 같은 일. 정해진 순서의 그래프로 충분합니다.

먼저 에이전트 하나로 만들어 보고, 한계가 보일 때 나눕니다.

## 2. 라우터 vs 슈퍼바이저

**라우터**는 처음 받은 문의 문장만 보고 담당 워커를 골라 보내면 끝납니다(보통 한 명). **슈퍼바이저**는 워커가 들고 온 보고(예: "배송 완료 후 9일")를 읽고 다음 워커를 다시 정합니다.

라우터가 여러 담당에게 동시에 보내는 경우도 있지만, 워커의 보고를 읽고 다음 담당을 다시 고르지 않는다는 점은 같습니다.

CS 팀(주문 담당 order_worker, 규정 담당 policy_worker, 환불 담당 refund_worker, 답변 작성 reply_writer)이 고객 문의 "ORD-1024 블루투스 이어폰 한쪽이 안 들려요. 환불해 주세요."를 처리하는 예:

1. supervisor가 order_worker에게 ORD-1024 조회를 맡깁니다.
2. order_worker가 "배송 완료 후 9일, 개봉"이라고 보고합니다. 단순 변심 환불 기한(7일)이 지났습니다.
3. supervisor가 이 보고를 읽고 policy_worker에게 불량 예외 규정을 찾게 합니다.

라우터라면 "환불"이라는 낱말만 보고 refund_worker에게 바로 보내고 끝납니다. 날짜와 규정은 아무도 확인하지 않습니다.

<img src="images/02_router_vs_supervisor.png" width="1050" alt="같은 ORD-1024 문의를 라우터는 '환불' 낱말만 보고 refund_worker로 보내 끝내고, 슈퍼바이저는 order_worker의 '배송 완료 후 9일, 개봉' 보고를 읽고 policy_worker에게 불량 예외 규정을 찾게 한다">

| 구분 | 라우터 | 슈퍼바이저 |
|---|---|---|
| 어울리는 일 | 담당이 처음부터 분명한 문의 (예: ORD-1027 배송 조회) | 앞 보고를 봐야 다음 담당이 정해지는 업무 (예: 기한이 지났으면 불량 예외 규정 확인) |
| LLM 호출 | 적음 (고르기 1번) | 많음 (워커가 돌아올 때마다 고르기) |

슈퍼바이저는 워커에게 일을 맡기고, 워커는 보고를 들고 돌아옵니다. 고객 답변까지 나오면 `FINISH`로 마칩니다.

<img src="images/03_supervisor_loop.png" width="1050" alt="supervisor가 ORD-1024를 처리하며 ① order_worker(배송 9일, 개봉) ② policy_worker(불량 30일 이내, 사진 필요) ③ order_worker(불량 사진 2장) ④ refund_worker(89,000원 환불) ⑤ reply_writer(고객 답변) 순서로 일을 맡기고 보고를 받은 뒤 FINISH로 끝낸다">

이 교안의 문의 3건은 모두 "환불해 주세요"로 끝나지만 필요한 확인이 다릅니다. 5-6에서 같은 팀이 세 문의를 서로 다른 확인을 거쳐 환불 2건, 사진 요청 1건으로 처리하는 것을 확인합니다.

**슈퍼바이저와 ReAct 에이전트(`create_agent`)는 같은 것인가요?**

둘 다 "결과를 읽고 다음 행동을 다시 고르는" 반복을 합니다. 다른 점은 **무엇을 고르느냐**입니다.

1. ReAct 에이전트: 에이전트 하나가 모든 도구를 들고, 매번 **도구**를 고릅니다. 도구 결과는 전부 그 에이전트의 대화 기록에 쌓입니다.
2. 슈퍼바이저: 매번 **워커**(자기 도구를 가진 에이전트)를 고릅니다. 워커는 안에서 도구를 골라 일하고, 슈퍼바이저에게는 정리된 보고만 돌아옵니다.
3. 그래서 슈퍼바이저 팀은 반복이 두 겹입니다. 바깥은 워커를 고르는 반복, 안쪽은 워커마다 도구를 고르는 반복입니다.

<img src="images/13_react_vs_supervisor.png" width="1050" alt="ReAct 에이전트는 혼자 주문 조회·규정 검색·환불 도구를 들고 매번 도구를 고르며 결과를 대화 기록 하나에 쌓고, 슈퍼바이저는 주문 담당·규정 담당 같은 워커를 고르며 각 워커 안에서 다시 도구를 고르는 반복이 돌아 보고만 돌아오는 두 겹 구조">

## 3. 실습 준비

### 3-1. 모델과 경로 설정

53일차와 같은 방식으로 환경 변수를 읽고 모델을 준비합니다.

In [ ]:
# 실습에 필요한 라이브러리를 불러옵니다.
import os
import json
from pathlib import Path
from dotenv import find_dotenv, load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from IPython.display import Image, Markdown

# material_dir는 output이 있는 단원 폴더입니다.
material_dir = Path(".")
data_dir = material_dir / "data"
output_dir = material_dir / "output"
output_dir.mkdir(parents=True, exist_ok=True)

# 단원 폴더에서 가까운 .env를 찾아 API 키를 읽습니다.
load_dotenv(find_dotenv(usecwd=True))
# 단원 .env에 없는 공통 키는 저장소 최상위 .env에서 읽습니다. 이미 읽은 키는 덮어쓰지 않습니다.
load_dotenv(material_dir.resolve().parent / ".env")

# 답변 모델: .env에 OPENAI_CHAT_MODEL이 없으면 기본 모델을 씁니다.
model_name = os.getenv("OPENAI_CHAT_MODEL", "gpt-5.6-luna")
llm = ChatOpenAI(model=model_name, use_responses_api=True)
print("답변 모델:", model_name)

### 3-2. 쇼핑몰 CS 팀 자료

이 교안의 예시는 **쇼핑몰 고객 문의(CS) 환불·교환 처리 팀**입니다. 주문 6건, 불량 접수 2건, 환불·교환 규정 5개를 딕셔너리와 리스트로 노트북 안에 둡니다.

환불 여부는 문의 문장만으로 정할 수 없습니다. 배송 후 지난 날수, 개봉 여부, 불량 증빙을 규정과 맞춰 봐야 합니다.

In [ ]:
from datetime import date

# 실습 기준일입니다. 배송 후 지난 날수를 이 날짜로 계산합니다.
TODAY = "2026-10-06"

# 주문 데이터: 주문번호별 상품·결제 금액·결제 수단·배송 완료일·개봉 여부입니다.
ORDERS = {
    "ORD-1021": {"상품": "무선 충전 패드 WP-10", "결제 금액": 29000, "결제 수단": "카드",
                 "배송 완료일": "2026-10-03", "개봉": "미개봉"},
    "ORD-1024": {"상품": "블루투스 이어폰 BE-200", "결제 금액": 89000, "결제 수단": "카드",
                 "배송 완료일": "2026-09-27", "개봉": "개봉"},
    "ORD-1026": {"상품": "노트북 거치대", "결제 금액": 42000, "결제 수단": "카드",
                 "배송 완료일": "2026-10-01", "개봉": "미개봉"},
    "ORD-1027": {"상품": "보조배터리 20000mAh", "결제 금액": 39000, "결제 수단": "네이버페이",
                 "배송 완료일": "2026-09-30", "개봉": "개봉"},
    "ORD-1031": {"상품": "무선 마우스 M-300", "결제 금액": 25000, "결제 수단": "카드",
                 "배송 완료일": "2026-10-03", "개봉": "미개봉"},
    "ORD-1040": {"상품": "전기 주전자 KT-15", "결제 금액": 39000, "결제 수단": "카카오페이",
                 "배송 완료일": "2026-09-24", "개봉": "개봉"},
}

# 불량 접수 데이터: 고객이 불량을 접수한 주문만 있습니다. 사진은 불량 증빙입니다.
DEFECT_REPORTS = {
    "ORD-1024": {"접수일": "2026-10-05", "증상": "왼쪽 이어버드에서 소리가 나지 않음", "사진": "2장 첨부"},
    "ORD-1040": {"접수일": "2026-10-05", "증상": "전원이 켜지지 않음", "사진": "없음"},
}

# 환불·교환 규정 데이터: 규정마다 id·제목·내용이 있습니다.
POLICIES = [
    {"id": "P1", "제목": "단순 변심 환불",
     "내용": "배송 완료일로부터 7일 이내, 미개봉 상품만 환불합니다."},
    {"id": "P2", "제목": "초기 불량 환불·교환",
     "내용": "배송 완료일로부터 30일 이내 초기 불량은 전액 환불 또는 교환합니다. 불량 사진 등 증빙이 접수되어야 합니다."},
    {"id": "P3", "제목": "개봉 상품",
     "내용": "개봉한 상품은 단순 변심 환불 대상이 아닙니다. 초기 불량은 개봉 여부와 관계없이 P2를 따릅니다."},
    {"id": "P4", "제목": "배송비 부담",
     "내용": "단순 변심 반품의 왕복 배송비 6,000원은 고객이 반품 접수 때 따로 결제하고, 환불은 결제 금액 전액입니다. 불량·오배송 배송비는 회사가 부담합니다."},
    {"id": "P5", "제목": "환불 처리 기간",
     "내용": "환불 승인 후 영업일 3~5일 안에 결제가 취소됩니다."},
]

## 4. Command: 다음 노드와 State에 쓸 값을 함께 반환

노드가 `Command`를 반환하면 두 가지를 한 번에 정합니다. `Command`는 51일차에 `interrupt`로 멈춘 그래프를 `Command(resume=...)`로 재개할 때 쓴 클래스입니다.

- `goto`: 다음에 실행할 노드 이름 (어디로 갈지)
- `update`: State에 쓸 값. 노드가 dict를 반환할 때와 똑같이 State가 바뀝니다.

<img src="images/04_command_goto.png" width="1050" alt="ORD-1024 환불 문의를 조건부 엣지에서는 classify가 category를 남기고 route_by_category가 policy_desk를 고르며, Command에서는 classify_and_go가 goto: policy_desk와 update를 함께 보낸다">

### 4-1. 조건부 엣지로 창구 나누기

LLM 없이 고객 문의를 주문 창구(배송·주문 문의)와 환불·교환 창구로 나누는 작은 그래프입니다. 먼저 익숙한 조건부 엣지로 만듭니다.

<img src="images/node_flow/l1_desk_edge_overview.png" width="1050" alt="classify 다음에 route_by_category가 order_desk 또는 policy_desk를 고르고, 두 창구 모두 END로 끝나는 그래프">

**State:** 문의(`question`), 문의 종류(`category`), 주문번호(`order_id`), 창구가 받은 값(`answer`) 필드 네 개입니다. 주문번호는 분류 노드가 정규 표현식 `ORD-\d+`로 문의 문장에서 찾습니다.

In [ ]:
# 정규 표현식(re), 타입 힌트, 그래프와 Command를 불러옵니다.
import re
from typing import Literal
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command

# [State] 창구 그래프의 공유 데이터: 노드가 읽고 쓰는 필드 네 개입니다.
class DeskState(TypedDict):
    question: str  # 고객 문의
    category: str  # 문의 종류(배송·주문 / 환불·교환). classify가 씁니다
    order_id: str  # 문의에서 찾은 주문번호. classify가 씁니다
    answer: str  # 창구가 읽은 문의 종류·주문번호. 창구 노드가 씁니다

**문의 분류 (`classify`)**

문의 종류와 주문번호를 State에 씁니다. 다음 노드는 라우팅 함수 `route_by_category`가 문의 종류를 읽어 따로 고릅니다.

<img src="images/node_flow/l1_desk_edge_classify.png" width="1050" alt="창구 그래프에서 지금 만드는 노드: classify">

In [ ]:
# [노드] 문의 분류
def classify(state: DeskState):
    """문의 종류와 주문번호를 State에 씁니다."""
    question = state["question"]

    # 'ORD-' 뒤에 숫자가 이어지는 부분을 주문번호로 찾습니다 (예: 'ORD-1024').
    order_id = re.search(r"ORD-\d+", question).group()

    # 환불·교환 낱말이 있으면 환불·교환 문의, 아니면 배송·주문 문의입니다.
    if "환불" in question or "교환" in question:
        category = "환불·교환"
    else:
        category = "배송·주문"

    # State에 쓸 값만 반환합니다. 어디로 갈지는 아래 라우팅 함수가 정합니다.
    return {"category": category, "order_id": order_id}


# [라우팅 함수] 문의 종류로 다음 창구 고르기
def route_by_category(state: DeskState) -> Literal["order_desk", "policy_desk"]:
    """classify가 적은 문의 종류를 보고 다음 노드 이름을 반환합니다."""
    # 반환한 이름이 곧 다음에 실행할 노드입니다.
    if state["category"] == "환불·교환":
        return "policy_desk"
    return "order_desk"

**주문 창구 (`order_desk`)**: `classify`가 State에 쓴 문의 종류와 주문번호를 읽어 `answer`에 적습니다.

<img src="images/node_flow/l1_desk_edge_order_desk.png" width="1050" alt="창구 그래프에서 지금 만드는 노드: order_desk">

In [ ]:
# [노드] 주문 창구
def order_desk(state: DeskState):
    """앞 노드가 State에 쓴 문의 종류와 주문번호를 읽어 answer에 적습니다."""
    # 읽은 값을 answer에 적어, 앞 노드가 쓴 값이 이 노드까지 전달됐는지 확인합니다.
    return {"answer": f"order_desk가 받은 값: {state['category']} {state['order_id']}"}

**환불·교환 창구 (`policy_desk`)**: `classify`가 State에 쓴 문의 종류와 주문번호를 읽어 `answer`에 적습니다.

<img src="images/node_flow/l1_desk_edge_policy_desk.png" width="1050" alt="창구 그래프에서 지금 만드는 노드: policy_desk">

In [ ]:
# [노드] 환불·교환 창구
def policy_desk(state: DeskState):
    """앞 노드가 State에 쓴 문의 종류와 주문번호를 읽어 answer에 적습니다."""
    # 읽은 값을 answer에 적어, 앞 노드가 쓴 값이 이 노드까지 전달됐는지 확인합니다.
    return {"answer": f"policy_desk가 받은 값: {state['category']} {state['order_id']}"}

**그래프 조립:** `add_conditional_edges`로 `classify` 뒤의 분기를 등록합니다.

In [ ]:
# [그래프 조립] 노드 셋을 등록하고 시작·분기·끝 연결을 겁니다.
edge_builder = StateGraph(DeskState)
edge_builder.add_node("classify", classify)
edge_builder.add_node("order_desk", order_desk)
edge_builder.add_node("policy_desk", policy_desk)

edge_builder.add_edge(START, "classify")
# classify가 끝나면 route_by_category가 반환한 이름의 노드로 갑니다.
edge_builder.add_conditional_edges("classify", route_by_category, ["order_desk", "policy_desk"])

# 두 창구 노드는 실행 뒤 END로 가서 그래프를 마칩니다.
edge_builder.add_edge("order_desk", END)
edge_builder.add_edge("policy_desk", END)

edge_app = edge_builder.compile()

# 컴파일한 노드와 연결을 이미지로 확인합니다.
# 점선(조건부 이동)은 add_conditional_edges에 넘긴 후보 노드 목록으로 그립니다.
display(Image(edge_app.get_graph().draw_mermaid_png()))

### 4-2. Command로 창구 나누기

분류와 이동을 한 노드에서 합니다. 라우팅 함수가 없고, `classify`에서 나가는 엣지도 등록하지 않습니다. 창구 노드 둘은 4-1의 것을 그대로 씁니다.

<img src="images/node_flow/l1_desk_command_overview.png" width="1050" alt="classify가 Command(goto)로 order_desk 또는 policy_desk를 고르고, 두 창구 모두 END로 끝나는 그래프">

**문의 분류와 이동 (`classify_and_go`)**

그래프에는 4-1과 같은 노드 이름 `classify`로 등록합니다. `update`의 문의 종류와 주문번호가 먼저 State에 쓰이고, 그다음 `goto`의 창구가 실행됩니다. 그래서 창구 노드가 이 값을 읽고, 최종 결과에도 남습니다.

<img src="images/node_flow/l1_desk_command_classify.png" width="1050" alt="Command 창구 그래프에서 지금 만드는 노드: classify">

In [ ]:
# [노드] 문의 분류와 창구 이동
# 반환 타입 힌트에 갈 수 있는 노드 이름을 적습니다. 엣지를 등록하지 않아도 그래프 이미지에 이 연결이 그려집니다.
def classify_and_go(state: DeskState) -> Command[Literal["order_desk", "policy_desk"]]:
    """State에 쓸 값(update)과 다음 창구(goto)를 함께 반환합니다."""
    question = state["question"]

    # 'ORD-' 뒤에 숫자가 이어지는 부분을 주문번호로 찾습니다 (예: 'ORD-1024').
    order_id = re.search(r"ORD-\d+", question).group()

    # 환불·교환 낱말이 있으면 환불·교환 창구, 아니면 주문 창구입니다.
    if "환불" in question or "교환" in question:
        next_desk = "policy_desk"
        category = "환불·교환"
    else:
        next_desk = "order_desk"
        category = "배송·주문"

    # update: State에 쓸 값 (dict를 반환할 때처럼 State가 바뀝니다) / goto: 그다음 실행할 노드
    return Command(goto=next_desk, update={"category": category, "order_id": order_id})

In [ ]:
# [그래프 조립] classify 자리에 classify_and_go를 넣습니다. 창구 노드는 조건부 엣지 그래프와 같은 함수를 씁니다.
command_builder = StateGraph(DeskState)
command_builder.add_node("classify", classify_and_go)
command_builder.add_node("order_desk", order_desk)
command_builder.add_node("policy_desk", policy_desk)

command_builder.add_edge(START, "classify")
# classify에서 나가는 엣지는 등록하지 않습니다. 반환한 Command가 길을 정합니다.
command_builder.add_edge("order_desk", END)
command_builder.add_edge("policy_desk", END)

command_app = command_builder.compile()

# 컴파일한 노드와 연결을 이미지로 확인합니다.
# 엣지를 등록하지 않았는데 점선이 보이는 이유: 노드 함수의 반환 타입 힌트
# Command[Literal[...]]에 적힌 이름을 갈 수 있는 곳으로 읽어 그립니다.
display(Image(command_app.get_graph().draw_mermaid_png()))

**실행 확인:** 같은 문의를 두 그래프에 넣어 창구가 받은 값을 비교합니다. 두 방식 모두 창구가 문의 종류와 주문번호를 받아야 합니다.

In [ ]:
# 같은 문의를 두 그래프에 넣고 창구가 받은 값을 나란히 비교합니다.
desk_questions = ["ORD-1027 배송지를 바꿀 수 있나요?", "ORD-1024 이어폰 환불해 주세요."]

for question in desk_questions:
    # category·order_id를 빈 값으로 시작합니다. 창구에 찍힌 값은 분류 노드가 쓴 값입니다.
    desk_input = {"question": question, "category": "", "order_id": "", "answer": ""}

    edge_result = edge_app.invoke(desk_input)
    command_result = command_app.invoke(desk_input)

    print("조건부 엣지 ->", edge_result["answer"], "| Command ->", command_result["answer"])

5절의 슈퍼바이저도 `Command`를 반환합니다. `goto`로 다음 워커를, `update`로 그 워커에게 줄 지시를 한 번에 정합니다.

| 구분 | 조건부 엣지 | Command |
|---|---|---|
| 다음 노드를 정하는 곳 | 노드 밖 라우팅 함수 `route_by_category` | 노드 `classify_and_go`의 반환값 `goto` |
| State에 쓸 값 | 노드가 `{"category": ..., "order_id": ...}`를 반환 | 같은 값을 `update`에 담아 `goto`와 함께 반환 |
| 어울리는 곳 | 규칙으로 갈 곳이 정해질 때 (예: "환불" 낱말로 창구 나누기) | 노드 안의 LLM 판단이 곧 다음 노드일 때 (예: supervisor가 policy_worker를 고름) |

## 5. CS 팀 슈퍼바이저 만들기

CS 팀을 담당별 도구, 워커 에이전트, 팀 상태, 슈퍼바이저 노드, 워커 노드, 그래프 조립과 실행, 처리 상한 순서로 만듭니다.

### 5-1. 담당별 도구

**워커**는 한 가지 일을 맡는 에이전트입니다. 워커마다 **그 일에 필요한 도구만** 주어, 권한 밖의 일(예: 답변 담당의 환불 실행)은 아예 할 수 없게 합니다.

예: 환불 요청이 답변 담당에게 잘못 간 경우

1. 돈이 나가는 환불 도구와 상품이 다시 나가는 교환 도구는 환불 담당(refund_worker)에게만 줍니다.
2. 답변 담당(reply_writer)에게 "89,000원 환불도 처리해 주세요"라고 시켜도, 답변 담당이 가진 도구는 답변 양식 하나뿐이라 환불을 실행할 방법이 없습니다. "환불은 할 수 없다"고 답할 뿐입니다.
3. 그래서 환불은 supervisor가 환불 담당에게 맡겼을 때만 실행됩니다. 맡기기 전에 주문·규정·증빙을 확인하게 하는 일은 5-4의 처리 원칙이 맡습니다.

5-2에서 답변 담당에게 실제로 환불을 시켜 보고 확인합니다.

<img src="images/05_tool_permission.png" width="1050" alt="워커마다 자기 도구만 든다. issue_refund는 refund_worker만 가지고, reply_writer는 '89,000원 환불도 처리해 주세요' 요청 쪽지를 받아도 issue_refund가 없어 '환불은 제 도구로 할 수 없어요'라고 답한다">

| 워커 | 맡은 일 | 도구 |
|---|---|---|
| order_worker | 주문·배송, 불량 접수 조회 | `get_order`, `get_defect_report` |
| policy_worker | 환불·교환 규정 검색 | `search_policy` |
| refund_worker | 환불 실행, 교환 접수 | `issue_refund`, `request_exchange` |
| reply_writer | 고객 답변 작성 | `get_reply_template` |

`issue_refund`는 실행할 때마다 `REFUND_LOG`에 기록을 남겨, 실제로 돈이 나갔는지 뒤에서 확인할 수 있게 합니다.

In [ ]:
# 도구 데코레이터, AI 메시지 클래스, 에이전트 생성 함수를 불러옵니다.
from langchain_core.tools import tool
from langchain_core.messages import AIMessage
from langchain.agents import create_agent

# 주문 담당 도구: 주문·배송과 불량 접수를 조회만 합니다.
@tool
def get_order(order_id: str) -> dict:
    """주문 상품·결제 금액·배송 완료일·개봉 여부와 배송 후 지난 날수를 조회합니다."""
    # 없는 주문번호면 안내 문장을 돌려줍니다.
    if order_id not in ORDERS:
        return {"안내": "없는 주문번호입니다."}

    # 원본 주문 데이터가 바뀌지 않게 복사본에 값을 더합니다.
    order = dict(ORDERS[order_id])

    # 규정의 기간 조건(7일·30일)과 비교할 수 있게, 배송 완료일부터 TODAY까지 지난 날수를 더합니다.
    delivered = date.fromisoformat(order["배송 완료일"])
    order["배송 후 지난 날수"] = (date.fromisoformat(TODAY) - delivered).days

    return order


@tool
def get_defect_report(order_id: str) -> dict:
    """주문의 불량 접수 내용(접수일·증상·사진 첨부 여부)을 조회합니다."""
    # 불량 접수가 없는 주문이면 안내 문장을 돌려줍니다.
    if order_id not in DEFECT_REPORTS:
        return {"안내": "접수된 불량 신고가 없습니다."}

    return DEFECT_REPORTS[order_id]

In [ ]:
# 규정 담당 도구: 규정 검색만 합니다. 뜻이 아니라 제목·내용에 낱말이 그대로 들어 있는지만 봅니다.
@tool
def search_policy(keyword: str) -> list[dict]:
    """제목이나 내용에 keyword가 들어간 환불·교환 규정을 찾습니다. keyword는 '불량'처럼 짧은 낱말 하나입니다."""
    # 제목이나 내용에 낱말이 들어 있는 규정만 모아 돌려줍니다.
    found = []
    for policy in POLICIES:
        if keyword in policy["제목"] or keyword in policy["내용"]:
            found.append(policy)

    return found

In [ ]:
# issue_refund가 실행될 때마다 한 건씩 쌓이는 환불 기록입니다. 문의를 처리하기 전에 비웁니다.
REFUND_LOG = []

# 환불 담당 도구: 실제로 돈이 나가거나 상품이 다시 나가는 작업입니다. 환불 담당 워커만 받습니다.
@tool
def issue_refund(order_id: str, reason: str) -> str:
    """주문의 결제 금액 전액을 환불합니다. 실제로 돈이 나가는 작업입니다."""
    # 금액은 주문 데이터에서 가져오고, 나간 환불을 REFUND_LOG에 남깁니다.
    amount = ORDERS[order_id]["결제 금액"]
    REFUND_LOG.append({"order_id": order_id, "amount": amount, "reason": reason})

    return f"환불 승인: {order_id} {amount:,}원, 사유: {reason}"


@tool
def request_exchange(order_id: str) -> str:
    """같은 상품의 새 제품으로 교환을 접수합니다."""
    # 교환은 환불 기록에 남기지 않고 접수 문장만 돌려줍니다.
    return f"교환 접수: {order_id} 새 제품 발송, 회수 기사 1~2일 안에 방문"

In [ ]:
# 답변 담당 도구: 답변 양식만 돌려줍니다. 주문·규정 같은 사실은 전달받은 보고에서 가져옵니다.
@tool
def get_reply_template() -> str:
    """고객 답변의 양식을 반환합니다."""
    return """1. 문의 확인 인사 한 줄
2. 확인한 주문 정보(주문번호·상품·배송 완료일)
3. 적용한 규정과 처리 결과(환불·교환, 금액, 처리 기간)
4. 고객이 할 일 또는 추가 안내"""

### 5-2. 워커 에이전트 만들기

`create_agent`로 워커 넷을 만듭니다. 모델은 같고 도구와 지시만 다릅니다.

In [ ]:
# 워커마다 그 일에 필요한 도구만 묶습니다.
order_tools = [get_order, get_defect_report]
policy_tools = [search_policy]
refund_tools = [issue_refund, request_exchange]
reply_tools = [get_reply_template]

# 모델은 같고 도구와 지시만 다릅니다.
# 주문 담당 워커: 조회한 항목을 빠짐없이 보고합니다.
order_agent = create_agent(llm, tools=order_tools, system_prompt=(
    "쇼핑몰 CS 팀의 주문 담당입니다. 지시받은 항목만 도구로 조회하고, "
    "도구가 돌려준 항목(배송 후 지난 날수·개봉 여부 포함)을 빠짐없이 짧게 보고하세요."))

# 규정 담당 워커: 짧은 낱말로 규정을 찾고 조건까지 보고합니다.
policy_agent = create_agent(llm, tools=policy_tools, system_prompt=(
    "쇼핑몰 CS 팀의 규정 담당입니다. 도구로 찾은 규정의 id와 내용을 근거로 짧게 보고하세요. "
    "search_policy는 낱말이 들어 있는지만 보므로 '불량', '변심', '개봉'처럼 짧은 낱말 하나로 검색하세요. "
    "규정이 요구하는 조건(기간·증빙 등)을 빠짐없이 적고, 찾지 못하면 다른 낱말로 다시 검색하세요."))

# 환불·교환 담당 워커: 지시받은 주문만 처리합니다.
refund_agent = create_agent(llm, tools=refund_tools, system_prompt=(
    "쇼핑몰 CS 팀의 환불·교환 담당입니다. 지시받은 주문으로 환불 또는 교환을 처리하고 결과를 짧게 보고하세요."))

# 답변 작성 워커: 보고에 있는 사실만으로 답변을 씁니다.
reply_agent = create_agent(llm, tools=reply_tools, system_prompt=(
    "쇼핑몰 CS 팀의 답변 작성 담당입니다. 양식을 도구로 확인하고, 전달받은 보고에 있는 사실만으로 "
    "고객에게 보낼 한국어 답변을 쓰세요. 금액·기간·날짜는 보고에 적힌 값만 쓰고, 보고에 없으면 숫자를 지어내지 말고 빼세요. "
    "도구로 할 수 없는 일은 할 수 없다고 답하세요."))

# 워커마다 받은 도구 이름을 확인합니다.
worker_tools = {
    "order_agent": order_tools,
    "policy_agent": policy_tools,
    "refund_agent": refund_tools,
    "reply_agent": reply_tools,
}

for agent_name, tools in worker_tools.items():
    # 도구 객체에서 이름만 꺼냅니다.
    names = []
    for each_tool in tools:
        names.append(each_tool.name)

    print(agent_name, "도구:", names)

**실행 확인:** 답변 담당에게 환불 처리까지 부탁합니다. `issue_refund`가 없으므로 환불은 하지 못하고, 할 수 없다고 답해야 합니다.

In [ ]:
# reply_agent에게 환불까지 시킵니다. 환불 도구(issue_refund)가 없어 실행할 수 없어야 합니다.
reply_check = reply_agent.invoke({"messages": [{"role": "user", "content": (
    "ORD-1024 이어폰 89,000원 환불을 지금 처리하고, 고객 답변도 써 주세요.")}]})

# 에이전트가 실제로 부른 도구를 메시지 기록에서 찾습니다.
for message in reply_check["messages"]:
    if isinstance(message, AIMessage):
        for call in message.tool_calls:
            print("도구 호출:", call["name"], call["args"])

# 마지막 메시지가 에이전트의 최종 답변입니다.
display(Markdown(reply_check["messages"][-1].text))

### 5-3. 팀 상태

슈퍼바이저는 `results`에 쌓인 워커 보고를 읽고 다음 워커와 지시를 `Command(goto, update)`로 보냅니다. 워커는 지금까지의 보고에 새 보고를 붙인 `results` 목록 전체를 `update`에 담아 `Command(goto="supervisor", update=...)`로 돌아옵니다. 워커 노드는 안에서 5-2의 `create_agent` 에이전트를 실행합니다.

<img src="images/node_flow/l1_team_overview.png" width="1050" alt="supervisor가 order_worker·policy_worker·refund_worker·reply_writer 중 하나에 일을 맡기고, 워커는 supervisor로 돌아오며, FINISH이면 END로 끝나는 그래프">

워커끼리는 직접 대화하지 않습니다. 지시(`instruction`)와 보고(`results`)를 State에 남겨 전달합니다.

In [ ]:
# [State] 슈퍼바이저 팀 그래프의 공유 데이터
class TeamState(TypedDict):
    # 처음 설정: 노드에서 바꾸지 않는 입력·한도
    request: str  # 고객 문의 원문
    max_turns: int  # 처리 상한: 워커에게 일을 맡길 수 있는 횟수. 닿으면 reply_writer에게 상담원 연결 답변을 1번 더 맡깁니다

    # supervisor가 쓰는 필드: 다음 워커에게 줄 지시와 방문 기록
    instruction: str  # 이번 워커에게 준 지시
    visits: list[str]  # 일을 맡긴 워커 순서
    turn: int  # 일을 맡긴 횟수

    # 워커가 쓰는 필드: 리듀서(새 값을 기존 값과 합치는 규칙)가 없어 자동으로 덧붙지 않고, 반환한 목록으로 통째로 바뀝니다.
    # 그래서 워커가 지금까지의 보고에 새 보고를 붙인 목록 전체를 돌려줍니다.
    results: list[dict]  # [{"worker": 워커 이름, "result": 보고}, ...]

### 5-4. 슈퍼바이저 노드 (`supervisor`)

지금까지 받은 보고를 LLM에 보여 주고, 다음 워커와 지시를 `Command`로 보냅니다.

<img src="images/node_flow/l1_team_supervisor.png" width="1050" alt="슈퍼바이저 그래프에서 지금 만드는 노드: supervisor">

`Route`는 다음 워커(`next`)와 지시(`instruction`)를 받는 구조화 출력입니다. `FINISH`를 고르면 `END`로 갑니다. `TEAM_RULES`는 팀의 처리 원칙입니다. 워커를 부르는 순서는 적지 않고 기준만 적습니다. 순서까지 정하면 고정 엣지로 짠 그래프와 다를 것이 없습니다.

1. 환불·교환은 주문 상태, 적용 규정, 규정이 요구하는 조건이 보고로 모두 확인된 뒤에만 맡깁니다.
2. 보고에 없는 사실은 그 사실을 알아낼 워커에게 맡기고, 이미 보고된 내용은 다시 시키지 않습니다.
3. 조건을 채우지 못하면 환불하지 않고 reply_writer가 고객이 할 일을 안내하게 합니다.
4. 고객 답변이 나오면 FINISH를 고릅니다. 이 조건이 없으면 답변이 나온 뒤에도 워커를 더 부릅니다.

같은 원칙에서도 보고 내용(배송 후 날수, 규정, 증빙)에 따라 부르는 워커가 달라집니다.

In [ ]:
# 워커 소개: 슈퍼바이저 프롬프트에 넣을 워커별 담당 업무입니다.
WORKER_GUIDE = (
    "order_worker: 주문·배송 정보 조회, 불량 접수(사진 첨부 여부) 조회. "
    "policy_worker: 환불·교환 규정 검색. "
    "refund_worker: 환불 실행, 교환 접수. "
    "reply_writer: 확인된 내용으로 고객 답변 작성."
)

# 팀의 처리 원칙: 워커를 부르는 순서는 정하지 않고 지켜야 할 기준만 적습니다.
# 누구를 언제 부를지는 슈퍼바이저가 지금까지의 보고를 읽고 정합니다.
TEAM_RULES = (
    "처리 원칙: "
    "1) 환불·교환은 되돌리기 어려우므로, 판단에 필요한 사실(주문 상태, 적용되는 규정, 그 규정이 요구하는 조건을 채웠는지)이 "
    "보고로 모두 확인된 뒤에만 refund_worker에게 맡깁니다. "
    "2) 판단에 필요한 사실이 아직 보고에 없으면, 그 사실을 알아낼 수 있는 워커에게 맡깁니다. 이미 보고된 내용은 다시 시키지 않습니다. "
    "3) 규정이 요구하는 조건을 채우지 못하면 환불·교환하지 말고, reply_writer에게 고객이 해야 할 일을 안내하게 합니다. "
    "4) 고객에게 보낼 답변이 나오면 FINISH를 고릅니다."
)

# 슈퍼바이저 판단의 출력 형식: 다음 워커(next)와 지시(instruction)
# next는 그래프의 노드 이름과 같아야 그대로 goto에 쓸 수 있습니다.
class Route(BaseModel):
    next: Literal["order_worker", "policy_worker", "refund_worker", "reply_writer", "FINISH"] = Field(
        description="다음에 일을 맡길 워커. reply_writer가 답변을 썼으면 FINISH")
    instruction: str = Field(
        description="다음 워커에게 줄 지시. 워커가 알아야 할 앞 보고(주문번호·날짜·규정)를 넣는다. FINISH이면 빈 문자열")

# 슈퍼바이저 프롬프트: 시스템 메시지에 워커 소개와 처리 원칙, 사람 메시지에 문의와 보고를 넣습니다.
supervisor_prompt = ChatPromptTemplate.from_messages([
    ("system", "쇼핑몰 CS 환불·교환 팀의 팀장입니다. 직접 조회하거나 처리하지 않고 워커에게 맡깁니다. "
     + WORKER_GUIDE + " " + TEAM_RULES +
     " 문의와 지금까지 보고를 읽고 다음 워커 한 명과 지시를 정하세요."),
    ("human", "고객 문의: {request}\n\n지금까지 보고:\n{results}"),
])

# 프롬프트 결과를 LLM에 넘겨 Route 형식으로 받습니다.
supervisor_chain = supervisor_prompt | llm.with_structured_output(Route, strict=True)

`format_results`는 보고 목록을 '[워커] 보고' 문단으로 바꿔 프롬프트에 넣는 보조 함수입니다. 노드 함수 앞부분의 `if` 문은 처리 상한 검사이며 5-7에서 다룹니다.

In [ ]:
# [보조 함수] LLM 컨텍스트 만들기: 워커 보고 목록을 supervisor 프롬프트와 워커에게 보내는 메시지에 넣을 '[워커] 보고' 문단으로 바꿉니다.
def format_results(results):
    """워커 보고 목록을 '[워커] 보고' 문단으로 묶습니다."""
    # 아직 보고가 없으면 첫 판단이라는 뜻으로 '아직 없음'을 넘깁니다.
    if not results:
        return "아직 없음"

    lines = []
    for row in results:
        lines.append(f"[{row['worker']}] {row['result']}")

    return "\n\n".join(lines)


# [노드] 슈퍼바이저
def supervisor(state: TeamState) -> Command[Literal["order_worker", "policy_worker",
                                                    "refund_worker", "reply_writer", END]]:
    """지금까지 보고를 읽고 다음 워커와 지시를 정해 Command로 보냅니다."""
    # 처리 상한 검사: turn이 max_turns에 닿으면 조회·환불은 더 맡기지 않고 상담원 연결 답변으로 마무리합니다.
    if state["turn"] >= state["max_turns"]:
        # reply_writer가 이미 답변을 썼으면 바로 끝냅니다.
        if "reply_writer" in state["visits"]:
            return Command(goto=END)

        # 아직 답변이 없으면 reply_writer에게 상담원 연결 답변을 맡깁니다.
        return Command(goto="reply_writer", update={
            "instruction": ("처리 상한에 닿았습니다. 환불·교환을 약속하지 말고, 지금까지 확인한 내용과 함께 '상담원 확인 필요'로 안내하세요. "
                            "확인하지 못한 항목은 상담원이 확인한다고만 쓰고, 고객에게 새로 요구하지 마세요."),
            "visits": state["visits"] + ["reply_writer"],
            "turn": state["turn"] + 1,
        })

    # 매번 문의와 지금까지 보고를 함께 보여 주어 앞 보고로 다음 일을 정하게 합니다.
    route = supervisor_chain.invoke({
        "request": state["request"],
        "results": format_results(state["results"]),
    })

    # FINISH를 고르면 State를 바꾸지 않고 끝냅니다.
    if route.next == "FINISH":
        return Command(goto=END)

    # goto: 고른 워커 / update: 그 워커가 읽을 지시와 방문 기록·턴 수
    return Command(goto=route.next, update={
        "instruction": route.instruction,
        "visits": state["visits"] + [route.next],
        "turn": state["turn"] + 1,
    })

**실행 확인:** 문의 3건을 정하고, ORD-1024로 슈퍼바이저 노드를 직접 호출해 첫 판단을 봅니다.

In [ ]:
# 고객 문의 데이터: 주문번호별 문의 문장 3건입니다. 모두 "환불해 주세요"로 끝납니다.
TICKETS = {
    "ORD-1031": "ORD-1031 단순 변심이에요. 환불해 주세요.",
    "ORD-1024": "ORD-1024 블루투스 이어폰 한쪽이 안 들려요. 환불해 주세요.",
    "ORD-1040": "ORD-1040 전기 주전자가 고장 났어요. 환불해 주세요.",
}

In [ ]:
# 첫 판단용 입력: 보고·방문 기록은 비우고 turn은 0에서 시작합니다.
first_input = {
    "request": TICKETS["ORD-1024"],
    "max_turns": 8,
    "instruction": "",
    "visits": [],
    "turn": 0,
    "results": [],
}

# 그래프 없이 노드 함수만 호출해 첫 판단을 봅니다. 반환된 Command는 실행되지 않고 내용만 확인합니다.
first_step = supervisor(first_input)
print("다음 노드:", first_step.goto)

# 슈퍼바이저가 그 워커에게 준 지시입니다.
print("지시:", first_step.update["instruction"])

### 5-5. 워커 노드

`run_worker`는 고객 문의, 지시, 지금까지 받은 보고(`results`)를 워커 에이전트에 전달하고, 새 보고를 덧붙인 `results`를 반환합니다. 네 워커 노드가 함께 씁니다.

- **부모 그래프와 자식 그래프:** `create_agent`로 만든 워커 에이전트도 하나의 그래프입니다. 팀 그래프의 노드 안에서 워커 에이전트를 실행하면 팀 그래프가 부모, 워커 에이전트가 자식 그래프(서브그래프)가 됩니다. 워커 에이전트는 메시지 기록 전체를 돌려주지만, `run_worker`는 마지막 보고 문장만 꺼내 팀 State의 `results`에 담습니다.
- **기존 보고까지 반환하는 이유:** `results` 필드에는 리듀서(50일차 `add_messages`처럼 새 값을 기존 값과 합치는 규칙)가 없어서 노드가 반환한 리스트가 필드를 통째로 바꿉니다. 그래서 기존 보고에 새 보고를 덧붙인 리스트 전체를 반환합니다. 슈퍼바이저가 한 번에 워커 한 명만 보내므로 같은 필드에 동시에 쓰는 일이 없어, 리듀서 없이도 문제가 생기지 않습니다.

In [ ]:
# [보조 함수] 워커 실행: 고객 문의·지시·지금까지의 보고로 워커 에이전트의 LLM 컨텍스트를 만들고, 실행한 뒤 기존 보고에 새 보고를 붙인 새 목록을 반환합니다.
def run_worker(agent, worker, state):
    """고객 문의·지시·지금까지 보고를 워커 에이전트에 전달하고, 새 보고를 덧붙인 results를 반환합니다."""
    # LLM 컨텍스트: 고객 문의, 이번 지시, 팀이 지금까지 확인한 내용. 워커끼리는 직접 대화하지 않으므로 앞 보고를 여기에 넣어 줍니다.
    message = (f"고객 문의: {state['request']}\n\n지시: {state['instruction']}\n\n"
               f"팀이 지금까지 확인한 내용:\n{format_results(state['results'])}")

    reply = agent.invoke({"messages": [{"role": "user", "content": message}]})

    # 마지막 메시지(워커의 최종 보고)만 꺼내 기존 보고 뒤에 붙인 새 목록을 반환합니다.
    return state["results"] + [{"worker": worker, "result": reply["messages"][-1].text}]

**주문 담당 워커 (`order_worker`)**: `order_agent`에 조회를 맡기고 `supervisor`로 돌아갑니다.

<img src="images/node_flow/l1_team_order_worker.png" width="1050" alt="슈퍼바이저 그래프에서 지금 만드는 노드: order_worker">

In [ ]:
# [노드] 주문 담당 워커
def order_worker(state: TeamState) -> Command[Literal["supervisor"]]:
    """order_agent의 보고를 results에 더하고 supervisor로 돌아갑니다."""
    # 워커 에이전트를 먼저 실행하고, 새 보고가 더해진 results를 update에 담습니다.
    results = run_worker(order_agent, "order_worker", state)
    return Command(goto="supervisor", update={"results": results})

**규정 담당 워커 (`policy_worker`)**: `policy_agent`에 규정 검색을 맡기고 `supervisor`로 돌아갑니다.

<img src="images/node_flow/l1_team_policy_worker.png" width="1050" alt="슈퍼바이저 그래프에서 지금 만드는 노드: policy_worker">

In [ ]:
# [노드] 규정 담당 워커
def policy_worker(state: TeamState) -> Command[Literal["supervisor"]]:
    """policy_agent의 보고를 results에 더하고 supervisor로 돌아갑니다."""
    # 워커 에이전트를 먼저 실행하고, 새 보고가 더해진 results를 update에 담습니다.
    results = run_worker(policy_agent, "policy_worker", state)
    return Command(goto="supervisor", update={"results": results})

**환불 담당 워커 (`refund_worker`)**: `refund_agent`가 환불·교환을 처리하고 `supervisor`로 돌아갑니다.

<img src="images/node_flow/l1_team_refund_worker.png" width="1050" alt="슈퍼바이저 그래프에서 지금 만드는 노드: refund_worker">

In [ ]:
# [노드] 환불 담당 워커
def refund_worker(state: TeamState) -> Command[Literal["supervisor"]]:
    """refund_agent의 보고를 results에 더하고 supervisor로 돌아갑니다."""
    # 워커 에이전트를 먼저 실행하고, 새 보고가 더해진 results를 update에 담습니다.
    results = run_worker(refund_agent, "refund_worker", state)
    return Command(goto="supervisor", update={"results": results})

**답변 작성 워커 (`reply_writer`)**: `reply_agent`가 쓴 고객 답변을 들고 `supervisor`로 돌아갑니다. 끝낼지는 슈퍼바이저가 정합니다.

<img src="images/node_flow/l1_team_reply_writer.png" width="1050" alt="슈퍼바이저 그래프에서 지금 만드는 노드: reply_writer">

In [ ]:
# [노드] 답변 작성 워커
def reply_writer(state: TeamState) -> Command[Literal["supervisor"]]:
    """reply_agent의 보고를 results에 더하고 supervisor로 돌아갑니다."""
    # 워커 에이전트를 먼저 실행하고, 새 보고가 더해진 results를 update에 담습니다.
    results = run_worker(reply_agent, "reply_writer", state)
    return Command(goto="supervisor", update={"results": results})

### 5-6. 그래프 조립과 실행

엣지는 `START -> supervisor` 하나만 등록합니다. 실행 중 이동은 노드가 반환하는 `Command`의 `goto`가 정하고, 그래프 이미지의 점선은 반환 타입 힌트 `Command[Literal[...]]`를 읽어 그립니다.

In [ ]:
# [그래프 조립] 슈퍼바이저와 워커 넷을 노드로 등록합니다.
team_builder = StateGraph(TeamState)
team_builder.add_node("supervisor", supervisor)
team_builder.add_node("order_worker", order_worker)
team_builder.add_node("policy_worker", policy_worker)
team_builder.add_node("refund_worker", refund_worker)
team_builder.add_node("reply_writer", reply_writer)

# 시작 엣지만 등록합니다. 나머지 이동은 노드가 반환하는 Command가 정합니다.
team_builder.add_edge(START, "supervisor")

team_app = team_builder.compile()

# 컴파일한 노드와 연결을 이미지로 확인합니다.
# 엣지를 등록하지 않았는데 점선이 보이는 이유: 노드 함수의 반환 타입 힌트
# Command[Literal[...]]에 적힌 이름을 갈 수 있는 곳으로 읽어 그립니다.
display(Image(team_app.get_graph().draw_mermaid_png()))

문의 3건을 같은 팀에 차례로 넣습니다. 문의마다 환불 기록(`REFUND_LOG`)을 비우고 시작해, 실제로 환불이 나갔는지 함께 봅니다.

| 문의 | 자료 | 기대하는 처리 |
|---|---|---|
| ORD-1031 단순 변심 | 배송 3일, 미개봉 | 주문과 단순 변심 규정을 확인한 뒤 환불 |
| ORD-1024 이어폰 불량 | 배송 9일, 불량 사진 2장 | 불량 규정과 사진을 확인한 뒤 환불 |
| ORD-1040 주전자 고장 | 배송 12일, 불량 사진 없음 | 규정이 요구하는 사진이 없으므로 환불하지 않고 사진 요청 |

In [ ]:
# 문의마다 방문 순서·환불 기록·워커 보고·슈퍼바이저 판단 횟수를 모아 둡니다.
team_runs = {}
for ticket_id, request in TICKETS.items():
    # 문의마다 환불 기록을 비우고 새 입력으로 시작합니다.
    REFUND_LOG.clear()
    ticket_input = {
        "request": request,
        "max_turns": 8,
        "instruction": "",
        "visits": [],
        "turn": 0,
        "results": [],
    }

    result = team_app.invoke(ticket_input)

    # 슈퍼바이저 판단 횟수 = 워커에게 맡긴 횟수(turn) + 마치기로 한 판단 1번
    team_runs[ticket_id] = {
        "visits": result["visits"],
        "refunds": list(REFUND_LOG),
        "results": result["results"],
        "decisions": result["turn"] + 1,
    }

    print(ticket_id, "방문 순서:", " -> ".join(result["visits"]), "| 환불 기록:", len(REFUND_LOG), "건")

# 처리 기록을 output에 남겨 다른 상담원이 다시 읽을 수 있게 합니다.
runs_path = output_dir / "lesson01_team_runs.json"
runs_path.write_text(json.dumps(team_runs, ensure_ascii=False, indent=2), encoding="utf-8")
print("저장:", runs_path)

**확인:** ORD-1040은 refund_worker를 거치지 않고 끝납니다. ORD-1031과 ORD-1024는 방문 순서가 같을 수 있지만, 적용한 규정(P1 단순 변심, P2 초기 불량)과 확인한 증빙이 다릅니다. 어느 워커를 부를지는 미리 정해 두지 않고 슈퍼바이저가 보고를 읽고 정합니다. 아래 ORD-1024 보고에서 "배송 후 9일" 보고 뒤에 불량 규정을 찾게 했는지, 규정이 요구한 불량 사진을 확인한 뒤에 환불을 맡겼는지 봅니다.

In [ ]:
# ORD-1024의 워커 보고를 순서대로 읽습니다. 마지막 고객 답변은 빼고 봅니다.
for row in team_runs["ORD-1024"]["results"][:-1]:
    print(f"[{row['worker']}]", row["result"], "\n")

ORD-1024의 고객 답변입니다. 적용 규정과 환불 금액이 보고와 맞는지 봅니다.

In [ ]:
# ORD-1024의 마지막 보고가 reply_writer의 고객 답변입니다.
display(Markdown(team_runs["ORD-1024"]["results"][-1]["result"]))

ORD-1040의 고객 답변입니다. 환불을 약속하지 않고 불량 사진을 요청하는지 봅니다.

In [ ]:
# ORD-1040은 증빙이 없어 환불하지 않은 경우의 고객 답변입니다.
display(Markdown(team_runs["ORD-1040"]["results"][-1]["result"]))

### 5-7. 처리 상한과 상담원 연결

**처리 상한**은 슈퍼바이저가 워커에게 일을 맡길 수 있는 횟수(`max_turns`)입니다. `turn`이 상한에 닿으면 조회·환불은 더 맡기지 않고, reply_writer에게 상담원 연결 답변을 한 번만 맡긴 뒤 끝냅니다. 확인을 다 못 한 문의는 상담원이 이어받습니다.

ORD-1024를 `max_turns=2`로 처리할 때의 예:

1. 1턴: order_worker가 "배송 완료 후 9일, 개봉"을 보고합니다.
2. 2턴: policy_worker가 "초기 불량은 30일 이내, 증빙 필요"를 보고합니다.
3. 상한에 닿았으므로 supervisor는 refund_worker에게 환불을 맡기지 않습니다.
4. reply_writer가 환불을 약속하지 않고 "상담원 확인 필요"로 안내합니다. 답변이 나오면 supervisor는 바로 끝냅니다.

<img src="images/06_turn_guard.png" width="1050" alt="1턴 order_worker의 '배송 완료 후 9일, 개봉', 2턴 policy_worker의 '불량 30일 이내, 증빙 필요' 뒤 상한 2/2에 닿아 refund_worker는 막히고 reply_writer가 '상담원 확인 필요'로 넘긴다">

5-4의 `supervisor` 앞부분 `if` 문이 이 일을 합니다. 같은 팀 그래프에 상한만 2로 넣어 실행합니다.

In [ ]:
# 이번 실행에서 나간 환불만 보도록 환불 기록을 비웁니다.
REFUND_LOG.clear()

# 앞 실행과 같은 입력에서 처리 상한(max_turns)만 2로 줄입니다.
guard_input = {
    "request": TICKETS["ORD-1024"],
    "max_turns": 2,
    "instruction": "",
    "visits": [],
    "turn": 0,
    "results": [],
}

guard_result = team_app.invoke(guard_input)

print("방문 순서:", " -> ".join(guard_result["visits"]), "| 환불 기록:", len(REFUND_LOG), "건")

**확인:** 환불 기록이 0건이고, 답변이 상담원 확인으로 넘기는지 봅니다.

In [ ]:
# 상한 때문에 상담원에게 넘긴 답변입니다.
display(Markdown(guard_result["results"][-1]["result"]))

## 6. 워커를 도구로 부르는 슈퍼바이저

5절은 슈퍼바이저를 그래프 노드로 만들었습니다. 같은 팀을 **supervisor 에이전트 하나가 워커를 도구로 부르는 방식**으로도 만들 수 있습니다. LangChain 문서의 Subagents 패턴입니다.

<img src="images/node_flow/l1_lead_overview.png" width="1050" alt="supervisor_agent가 하나뿐인 도구 ask_worker를 부르고, ask_worker가 worker 인자로 고른 워커 에이전트(order_agent·policy_agent·refund_agent·reply_agent) 한 명을 실행해 보고 문장만 돌려주며, supervisor_agent가 최종 답을 내고 END로 끝나는 구조">

| 구분 | 5절 supervisor 노드 | 6절 supervisor 에이전트 |
|---|---|---|
| 다음 담당을 정하는 곳 | `supervisor` 노드가 `Command(goto)`로 | `supervisor_agent`가 `ask_worker` 도구 호출로 |
| 만드는 코드 | State·노드·연결을 직접 설계 | 도구 함수 하나 + `create_agent` |
| 워커가 보는 것 | 고객 문의, 지시, 팀이 지금까지 확인한 보고 전체(`run_worker`가 넣어 줌) | supervisor 에이전트가 지시에 적어 준 내용만 |
| 상한에 닿았을 때 | 노드의 `if`가 다음 갈 곳을 정함 (예: 상담원 연결 답변을 맡기고 끝냄) | 미들웨어가 supervisor 에이전트의 실행을 그 자리에서 끝냄. 끝난 뒤 무엇을 할지는 정할 수 없음 |
| 워커를 부르는 순서 | 필요하면 노드의 `if`로 막을 수 있음 (예: 확인 전에는 환불 담당으로 못 가게) | 프롬프트로 지시할 뿐, 지킬지는 모델이 정함 |
| 어울리는 일 | 순서·승인이 꼭 필요한 업무 (예: 확인 뒤 환불) | 워커 조합이 자주 바뀌고 빨리 만들어야 하는 업무 (예: 사내 문의 응대 보조) |

supervisor 에이전트는 한 응답에서 `ask_worker`를 여러 번 함께 부를 수 있습니다. 주문 조회와 규정 검색을 동시에 맡기면 빨라지지만, 확인이 끝나기 전에 환불까지 함께 맡길 수도 있습니다. 그래서 아래 코드는 모델에 `parallel_tool_calls=False`를 주어 한 응답에 도구를 하나만 부르게 합니다. 한 번에 한 워커씩 맡기게 됩니다.

**워커를 부르는 도구 (`ask_worker`)**: 워커는 다른 워커의 보고를 모르므로, 필요한 앞 보고는 supervisor 에이전트가 지시에 넣어야 합니다.

In [ ]:
# 워커 이름으로 워커 에이전트를 찾습니다.
WORKER_AGENTS = {
    "order_worker": order_agent,
    "policy_worker": policy_agent,
    "refund_worker": refund_agent,
    "reply_writer": reply_agent,
}

# ask_worker가 불릴 때마다 워커 이름을 쌓아 호출 순서를 남깁니다. 문의마다 비웁니다.
ASK_LOG = []

# [도구] 워커에게 일을 맡기는 도구
@tool
def ask_worker(worker: Literal["order_worker", "policy_worker", "refund_worker", "reply_writer"],
               instruction: str) -> str:
    """워커 한 명(worker)에게 지시(instruction)를 맡기고 워커의 최종 보고 문장만 돌려받습니다."""
    # 부른 워커를 기록하고, 지시 문장만 메시지로 보내 그 워커 에이전트를 실행합니다. 앞 보고는 지시에 적힌 만큼만 전달됩니다.
    ASK_LOG.append(worker)
    reply = WORKER_AGENTS[worker].invoke({"messages": [{"role": "user", "content": instruction}]})

    # 워커 안에서 부른 도구 기록은 버리고, 마지막 보고 문장만 돌려줍니다.
    return reply["messages"][-1].text

**supervisor 에이전트 (`supervisor_agent`)**: 워커 소개와 처리 원칙은 5절과 같습니다. 처리 상한은 `max_turns` 대신 21·22일차에 쓴 `ModelCallLimitMiddleware`로 둡니다.

In [ ]:
from langchain.agents.middleware import ModelCallLimitMiddleware

# 한 응답에 도구를 하나만 부르게 합니다. 확인이 끝나기 전에 환불 담당까지 함께 부르는 일을 막습니다.
supervisor_llm = ChatOpenAI(model=model_name, use_responses_api=True, model_kwargs={"parallel_tool_calls": False})

# supervisor 에이전트의 도구는 ask_worker 하나입니다. 워커 소개와 처리 원칙은 supervisor 노드와 같습니다.
supervisor_agent = create_agent(supervisor_llm, tools=[ask_worker], system_prompt=(
    "쇼핑몰 CS 환불·교환 팀의 팀장입니다. 직접 조회하거나 처리하지 않고 ask_worker로 워커에게 맡깁니다. "
    + WORKER_GUIDE + " " + TEAM_RULES +
    " 워커는 다른 워커의 보고를 모르므로 지시에 필요한 앞 보고(주문번호·날짜·규정)를 넣으세요. "
    "FINISH 대신, reply_writer의 고객 답변을 받으면 그 답변을 최종 답으로 내세요."),
    # 처리 상한: supervisor 에이전트의 모델 호출이 10번에 닿으면 멈춥니다. 워커 안의 모델 호출은 세지 않습니다.
    middleware=[ModelCallLimitMiddleware(run_limit=10, exit_behavior="end")])

**실행 확인:** 같은 문의 3건을 supervisor 에이전트로 처리하고, 워커를 부른 순서와 환불 기록을 봅니다.

In [ ]:
# [보조 함수] 출력용 문자열 만들기: 환불 기록을 '89,000원' 또는 '없음'으로 바꿉니다.
def refund_text(refunds):
    """환불 기록 목록을 '89,000원' 또는 '없음'으로 바꿉니다."""
    if not refunds:
        return "없음"

    # 환불 건마다 금액에 천 단위 쉼표를 넣어 이어 붙입니다.
    amounts = []
    for refund in refunds:
        amounts.append(f"{refund['amount']:,}원")

    return ", ".join(amounts)


# 같은 문의 3건을 supervisor 에이전트로 처리합니다.
supervisor_runs = {}
for ticket_id, request in TICKETS.items():
    # 문의마다 환불 기록과 워커 호출 기록을 비웁니다.
    REFUND_LOG.clear()
    ASK_LOG.clear()

    supervisor_reply = supervisor_agent.invoke({"messages": [{"role": "user", "content": request}]})

    # supervisor 에이전트의 마지막 메시지가 최종 답변입니다.
    supervisor_runs[ticket_id] = {
        "calls": list(ASK_LOG),
        "refunds": list(REFUND_LOG),
        "reply": supervisor_reply["messages"][-1].text,
    }

    print(ticket_id, "워커 호출 순서:", " -> ".join(ASK_LOG), "| 환불:", refund_text(REFUND_LOG))

ORD-1040에 대한 supervisor 에이전트의 답변입니다. 사진이 없으므로 환불하지 않고 사진을 요청하는지 봅니다.

In [ ]:
# ORD-1040에 대한 supervisor 에이전트의 최종 답변입니다.
display(Markdown(supervisor_runs["ORD-1040"]["reply"]))

## 7. 핵심 정리

- 라우터는 처음 문의 문장만 보고 워커를 한 번 고르면 끝납니다. 슈퍼바이저는 워커가 돌아올 때마다 보고(예: "배송 완료 후 9일")를 읽고 다음 워커를 다시 고릅니다. 그래서 같은 팀이 문의마다 다른 확인을 거쳐 다른 결과를 냅니다.
- `Command(goto, update)`의 `goto`는 다음 노드, `update`는 State에 쓸 값입니다(노드가 dict를 반환할 때처럼 State가 바뀝니다). 반환 타입 힌트 `Command[Literal[...]]`에는 갈 수 있는 노드를 적고, 그래프 이미지는 이 힌트로 연결을 그립니다.
- 워커마다 그 일에 필요한 도구만 줍니다. 환불처럼 돈이 나가는 도구는 한 워커에게만 줍니다.
- 슈퍼바이저와 워커를 오가는 반복 그래프에는 처리 상한(`max_turns`)을 두고, 상한에 닿으면 환불을 약속하지 말고 사람(상담원)에게 넘깁니다.
- 워커를 도구로 부르는 supervisor 에이전트로도 같은 팀을 만들 수 있습니다. supervisor 에이전트 방식에서는 워커가 다른 워커의 보고를 모르므로, 필요한 앞 보고를 지시에 넣어야 합니다. 순서를 코드로 막아야 하면 노드에 `if` 검사를 넣을 수 있는 그래프 방식을 씁니다.